<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_4_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.4 — Data Cleaning

## Learning Objectives

By the end of this notebook you should be able to:

- Detect and count missing values with `.isnull()` and `.notna()`
- Choose between dropping missing rows (`.dropna()`) and filling them (`.fillna()`)
- Find and remove duplicate rows
- Rename columns to be cleaner and more consistent
- Fix data types with `.astype()` and `pd.to_numeric()`
- Standardize and transform string columns with the `.str` accessor

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
raw = pd.read_csv(url)
raw.columns = raw.columns.str.lower()
raw = raw.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
print(raw.shape)
raw.head()

## Why This Step Exists

You've loaded the data, inspected it, and learned to filter it. Before you can trust any analysis, though, you need to ask: *is the data actually correct?*

Real datasets have problems. Missing values — a passenger's age wasn't recorded. Wrong types — a numeric column loaded as text. Inconsistent strings — `"Male"`, `"male"`, and `"MALE"` all meaning the same thing. Duplicate rows — the same record entered twice. Each of these problems will silently corrupt your analysis if you don't address it.

Data cleaning is unglamorous, but skipping it is the most common source of wrong answers in data analysis. A rough industry estimate is that analysts spend 60–80% of their time here.

## 1. Checking for Missing Values

The first question on any new dataset: *what's missing?*

This Titanic source is already clean — no missing values. That's unusual. Many other versions of this dataset have about 20% of ages missing (not all ships recorded every passenger's age). To make the cleaning tools meaningful, we'll introduce realistic gaps.

In [ ]:
# This version is complete — but let's verify
raw.isnull().sum()

In [ ]:
# Introduce ~20% missing ages, as seen in other Titanic sources
rng = np.random.default_rng(42)
df = raw.copy()
missing_idx = rng.choice(df.index, size=int(0.2 * len(df)), replace=False)
df.loc[missing_idx, "age"] = np.nan

# Now check
missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(1)
pd.DataFrame({"missing": missing_counts, "pct": missing_pct})

In [ ]:
# Which rows are affected? Use any(axis=1) — "does any column in this row have a NaN?"
rows_with_missing = df[df.isnull().any(axis=1)]
print(f"Rows with at least one missing value: {len(rows_with_missing)}")
rows_with_missing[["name", "age", "survived"]].head()

## 2. Deciding What to Do About Missing Data

Once you know what's missing, you have two choices: **drop** the rows with missing values, or **fill** them with something reasonable. Which you choose depends on two things:

1. *How much data would you lose?* Dropping 20% of rows is significant and may bias your results.
2. *Why is the data missing?* If it's random (some ages just weren't recorded), filling with the median is reasonable. If it's systematic (e.g., the poorest passengers were never asked), filling with a blanket value can introduce bias.

### Dropping with `.dropna()`

In [ ]:
df_dropped = df.dropna()
print(f"Original: {len(df)} rows")
print(f"After dropping any row with missing values: {len(df_dropped)} rows")
print(f"Lost: {len(df) - len(df_dropped)} rows ({(len(df)-len(df_dropped))/len(df):.0%})")

In [ ]:
# More surgical: drop only rows where 'age' is missing
# Other columns (if they had missing values) would still be kept
df_dropped_age = df.dropna(subset=["age"])
print(f"Rows after dropping only age-missing rows: {len(df_dropped_age)}")

### Filling with `.fillna()`

Often you'd rather fill than lose data. The most common approach for a numeric column like age is to fill with the **median** — the middle value — which is more robust to outliers than the mean.

In [ ]:
median_age = df["age"].median()
print(f"Median age (computed on non-missing rows): {median_age}")

df_filled = df.copy()
df_filled["age"] = df_filled["age"].fillna(median_age)
print(f"Missing after fill: {df_filled['age'].isnull().sum()}")

In [ ]:
# You can fill multiple columns at once with a dict — each column gets its own value
df_filled2 = df.fillna({
    "age": df["age"].median(),    # numeric: use median
    # add more columns here if they had missing values
})
df_filled2.isnull().sum()

> **Machine learning warning:** When you later split data into training and test sets, always compute the fill value (median, mean, mode) from the *training set only*, then apply that same value to the test set. Computing from the full dataset lets information from the test set leak into your model — a subtle but significant bug.

## 3. Duplicate Rows

Duplicates are less common than missing values but worth checking. They can appear when data from multiple sources is combined, or when a data entry process allows re-submission.

In [ ]:
print("Duplicates in the original dataset:", raw.duplicated().sum())

In [ ]:
# To demonstrate the tools, manufacture some duplicates
df_dupes = pd.concat([raw.head(5), raw.head(3)], ignore_index=True)
print(f"Rows including duplicates: {len(df_dupes)}")
print(f"Duplicate rows detected:   {df_dupes.duplicated().sum()}")

# Show which rows are duplicates
df_dupes[df_dupes.duplicated(keep=False)][["name", "age", "pclass"]]

In [ ]:
df_clean = df_dupes.drop_duplicates()
print(f"Rows after removing duplicates: {len(df_clean)}")

## 4. Renaming Columns

Column names from raw data are often inconsistent, long, or hard to type. Rename them early so the rest of your code is cleaner.

In [ ]:
# .rename() takes a dict: {old_name: new_name}
df_renamed = raw.rename(columns={
    "sibsp": "siblings_spouses",
    "parch": "parents_children",
    "pclass": "passenger_class",
})
print(df_renamed.columns.tolist())

## 5. Fixing Data Types

Some columns have the wrong type. `survived` is 0 or 1, but it's a *category*, not a number — you'd never add two survived-values together or take their average in a meaningful way. `pclass` is the same. Telling pandas these are categories makes operations on them more reliable and the code more self-documenting.

In [ ]:
print("Before:")
print(raw.dtypes)
print()
df_typed = raw.copy()
df_typed["survived"] = df_typed["survived"].astype("category")
df_typed["pclass"]   = df_typed["pclass"].astype("category")
df_typed["sex"]      = df_typed["sex"].astype("category")

print("After:")
print(df_typed.dtypes)

In [ ]:
# pd.to_numeric() handles columns that should be numeric but contain bad strings
# errors='coerce' converts unparseable values to NaN rather than crashing
messy_ages = pd.Series(["22", "38", "unknown", "35", "28"])
pd.to_numeric(messy_ages, errors="coerce")

## 6. Cleaning String Columns

The `name` column is a gold mine of hidden information — each name includes a title (Mr., Mrs., Miss., Master., Dr.) that encodes both sex and approximate age. But to use it, you first need to extract it cleanly.

In [ ]:
# Names look like: "Mr. Owen Harris Braund"
# Extract the first word ending in a period
titles = raw["name"].str.extract(r'^(\w+\.)')
titles.value_counts()

Those four titles cover almost everyone. `Master.` was the formal title for boys under roughly 14, so it's actually a proxy for age. `Miss.` was used for unmarried women of any age.

Fixing inconsistencies in string data is also common. Imagine the `sex` column had been entered by hand:

In [ ]:
# A messy sex column from manual data entry
messy = pd.Series(["  Male", "female", "MALE", "Female ", "male"])
cleaned = messy.str.strip().str.lower()
print(cleaned.unique())

## 7. A Complete Cleaning Workflow

Here is a realistic end-to-end cleaning function. It takes the raw CSV and produces a tidy, analysis-ready DataFrame.

In [ ]:
def prepare_titanic(raw_df, missing_age_seed=42):
    df = raw_df.copy()

    # Standardize column names
    df.columns = df.columns.str.lower()
    df = df.rename(columns={
        'siblings/spouses aboard': 'sibsp',
        'parents/children aboard': 'parch',
    })

    # Introduce realistic missing ages (~20%), then fill with median
    # (Skip this block if your source already has missing ages)
    rng = np.random.default_rng(missing_age_seed)
    missing_idx = rng.choice(df.index, size=int(0.2 * len(df)), replace=False)
    df.loc[missing_idx, "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())

    # Extract title from name, then drop name (too noisy to use directly)
    df["title"] = df["name"].str.extract(r'^(\w+\.)')
    df = df.drop(columns=["name"])

    # Derive a family feature
    df["has_family"] = ((df["sibsp"] > 0) | (df["parch"] > 0)).astype(int)

    # Fix data types
    for col in ["survived", "pclass", "sex", "title"]:
        df[col] = df[col].astype("category")

    return df

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
clean = prepare_titanic(pd.read_csv(url))

print(clean.dtypes)
print()
print(clean.isnull().sum())
print()
clean.head()

## Your Turn

1. In the `prepare_titanic` function above, why is the median computed *before* we split into training and test sets a potential problem for machine learning? How would you restructure it?

2. Create a new column `age_group` from the cleaned `df` with three categories: `"child"` (under 18), `"adult"` (18–60), `"senior"` (over 60). Use `.value_counts()` to see how many passengers fall in each group.

3. What does `pd.to_numeric(pd.Series(["7.25", "71.28", "bad", "53.10"]), errors="coerce")` return? What would happen without `errors="coerce"`? Try both.

4. The `title` column has five distinct values. Use boolean masking to find every passenger with the title `"Master."`. What is their average age? Does it confirm that `Master.` signals a child?

In [ ]:
# Your code here